# Telco Customer Churn Prediction — Capstone

This notebook explores the supplied dataset and compares four classification models. Run from the project folder.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
DATA=Path('Telco-Customer-Churn.csv')
df=pd.read_csv(DATA)
print(df.shape)
df.head()

## Data quality and target balance

In [ ]:
print(df.isna().sum())
print('Churn counts:\n',df['Churn'].value_counts())
print('Blank TotalCharges:', (df['TotalCharges'].astype(str).str.strip()=='').sum())

## Clean numeric charges and inspect churn patterns

In [ ]:
df['TotalCharges_num']=pd.to_numeric(df['TotalCharges'].replace(' ',np.nan),errors='coerce')
df.groupby('Churn')[['tenure','MonthlyCharges','TotalCharges_num']].mean().round(2)

In [ ]:
df.groupby('Contract')['Churn'].apply(lambda s:(s=='Yes').mean()*100).sort_values(ascending=False).plot(kind='bar',title='Observed churn rate by contract'); plt.ylabel('Churn rate (%)'); plt.tight_layout()

## Train/test split and model comparison

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score

model_df=df.drop(columns=['TotalCharges'])
model_df['TotalCharges_num']=df['TotalCharges_num']
y=(model_df.pop('Churn')=='Yes').astype(int)
X=model_df.drop(columns=['customerID'])
num=['SeniorCitizen','tenure','MonthlyCharges','TotalCharges_num']
cat=[c for c in X.columns if c not in num]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
prep=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]),cat)])
models={'Logistic Regression':LogisticRegression(max_iter=2000,class_weight='balanced',random_state=42),'Decision Tree':DecisionTreeClassifier(max_depth=6,min_samples_leaf=20,class_weight='balanced',random_state=42),'Random Forest':RandomForestClassifier(n_estimators=300,min_samples_leaf=3,class_weight='balanced_subsample',random_state=42,n_jobs=-1),'Gradient Boosting':GradientBoostingClassifier(random_state=42)}
rows=[]
for name,est in models.items():
 pipe=Pipeline([('prep',prep),('model',est)]); pipe.fit(X_train,y_train); pred=pipe.predict(X_test); proba=pipe.predict_proba(X_test)[:,1]
 rows.append([name,accuracy_score(y_test,pred),precision_score(y_test,pred),recall_score(y_test,pred),f1_score(y_test,pred),roc_auc_score(y_test,proba),average_precision_score(y_test,proba)])
results=pd.DataFrame(rows,columns=['Model','Accuracy','Precision','Recall','F1','ROC-AUC','PR-AUC']); results.round(4)